<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_12/01_fashion_mnist_adversarial_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fashion MNIST: Adversarial Examples & Adversarial Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_11/04_fashion_mnist_adversarial_training.ipynb)

Dieses Notebook baut auf `03_fashion_mnist_bias_variance.ipynb` auf und nutzt das dort trainierte CNN.

**Ziele dieser Übung:**
1. Verstehen, wie leicht sich ein gut trainiertes CNN mit minimalen, für das menschliche Auge kaum sichtbaren Störungen täuschen lässt (*Adversarial Examples*).
2. Den Robustheits-Einbruch quantitativ messen (Accuracy unter Angriff, in Abhängigkeit von der Störungsstärke `eps`).
3. Das Modell mittels *Adversarial Training* robuster gegen solche Angriffe machen und den Effekt messen.

Wir nutzen dafür die [Adversarial Robustness Toolbox (ART)](https://adversarial-robustness-toolbox.readthedocs.io/en/latest/), die native Keras/TensorFlow-Unterstützung bietet.

## 0. Setup

In [ ]:
# ART ist in Colab standardmaessig nicht vorinstalliert
!pip install adversarial-robustness-toolbox -q

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from functools import partial
import numpy as np
import os

from art.estimators.classification import KerasClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.defences.trainer import AdversarialTrainer

## 1. Vortrainiertes Modell laden

Wir laden dasselbe Modell, das in `03_fashion_mnist_bias_variance.ipynb` trainiert wurde, damit alle mit demselben Ausgangspunkt starten.

In [ ]:
model_url = "https://github.com/dgaida/wpf_dlml_th_public/raw/main/assets/exercises/week_11/fashion_mnist_bias_variance.keras"
keras_file = tf.keras.utils.get_file(fname="fashion_mnist_bias_variance.keras", origin=model_url)
print(f"Model downloaded to: {keras_file}")

## 2. Fashion MNIST laden und vorverarbeiten

Dieselbe Vorverarbeitung wie im Baseline-Notebook (Normalisierung auf `[0, 1]`, Kanal-Dimension hinzufuegen). Die Normalisierung auf `[0, 1]` ist wichtig: ART begrenzt Stoerungen ueber `clip_values`, die zum Werte-Bereich der Eingaben passen muessen.

In [ ]:
# 1. Fashion MNIST Datensatz laden
fashion_mnist = keras.datasets.fashion_mnist
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()

# 2. Vorverarbeitung
# Normalisierung und Hinzufuegen der Kanal-Dimension
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train_full = x_train_full[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Aufteilen in Trainings- und Validierungsdaten
x_train, x_valid = x_train_full[:55000], x_train_full[55000:]
y_train, y_valid = y_train_full[:55000], y_train_full[55000:]

class_names = ['T-Shirt/Top', 'Hose', 'Pullover', 'Kleid', 'Mantel',
               'Sandale', 'Hemd', 'Sneaker', 'Tasche', 'Stiefelette']

In [ ]:
model = keras.models.load_model(keras_file)
model.summary()

## 3. Modell in ART einbinden

ART-Angriffe arbeiten nicht direkt auf `tf.keras.Model`-Objekten, sondern auf einem Wrapper (`KerasClassifier`), der Framework-Details (Gradientenberechnung etc.) kapselt. Dadurch funktionieren dieselben Angriffe unabhaengig davon, ob das Modell in Keras, PyTorch oder einem anderen Framework implementiert ist.

`clip_values=(0, 1)` sagt ART, in welchem Wertebereich gueltige Eingaben liegen -- adversariale Stoerungen duerfen diesen Bereich nicht verlassen, sonst waeren es keine gueltigen Bilder mehr.

In [ ]:
classifier = KerasClassifier(model=model, clip_values=(0, 1), use_logits=False)

# Baseline: Clean Accuracy (ohne Angriff)
clean_preds = np.argmax(classifier.predict(x_test), axis=1)
clean_acc = np.mean(clean_preds == y_test)
print(f"Clean Accuracy (Testdaten, unveraendert): {clean_acc * 100:.2f} %")

## 4. Adversarial Examples mit FGSM erzeugen

Die **Fast Gradient Sign Method (FGSM)** verschiebt jedes Pixel um einen kleinen Betrag `eps` in die Richtung, die den Verlust des Modells am staerksten erhoeht:

$$x_{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$

Das Ergebnis: ein Bild, das fuer Menschen kaum von `x` zu unterscheiden ist, das Modell aber gezielt in die Irre fuehrt.

In [ ]:
eps = 0.1
attack_fgsm = FastGradientMethod(estimator=classifier, eps=eps)

# Angriff auf eine Teilmenge der Testdaten anwenden (schnelle Demo)
n_demo = 1000
x_test_subset = x_test[:n_demo]
y_test_subset = y_test[:n_demo]

x_adv_fgsm = attack_fgsm.generate(x=x_test_subset)

adv_preds = np.argmax(classifier.predict(x_adv_fgsm), axis=1)
adv_acc = np.mean(adv_preds == y_test_subset)
print(f"Accuracy unter FGSM-Angriff (eps={eps}): {adv_acc * 100:.2f} %")

### Visualisierung: Original vs. adversariales Bild

In [ ]:
def plot_adversarial_examples(x_clean, x_adv, y_true, preds_adv, class_names, max_n_to_show=5):
    """Zeigt Original-, Stoerungs- und adversariale Bilder nebeneinander,
    fokussiert auf falsch klassifizierte Beispiele.

    Args:
        x_clean: Array unveraenderter Eingabebilder, Form (N, 28, 28, 1).
        x_adv: Array adversarial veraenderter Bilder, gleiche Form wie
            x_clean.
        y_true: Wahre Klassen-Indizes der gezeigten Bilder.
        preds_adv: Vom Modell vorhergesagte Klassen-Indizes fuer x_adv.
        class_names: Liste der Klassennamen, indiziert nach Klassen-Index.
        max_n_to_show: Maximale Anzahl der darzustellenden falsch
            klassifizierten Beispiele.

    Returns:
        None. Die Funktion zeigt die Abbildung direkt mit plt.show() an.
    """
    # 1. Nur falsch klassifizierte Beispiele auswaehlen
    incorrect_indices = np.where(preds_adv != y_true)[0]
    n_incorrect = len(incorrect_indices)

    # 2. Anzahl der tatsaechlich darzustellenden Beispiele begrenzen
    n_to_plot = min(max_n_to_show, n_incorrect)

    if n_to_plot == 0:
        print("Keine falsch klassifizierten adversarialen Beispiele zum Anzeigen.")
        return

    x_clean_filtered = x_clean[incorrect_indices[:n_to_plot]]
    x_adv_filtered = x_adv[incorrect_indices[:n_to_plot]]
    y_true_filtered = y_true[incorrect_indices[:n_to_plot]]
    preds_adv_filtered = preds_adv[incorrect_indices[:n_to_plot]]

    fig, axes = plt.subplots(3, n_to_plot, figsize=(2.2 * n_to_plot, 7))

    # Stellen Sie sicher, dass 'axes' immer ein 2D-Array ist, auch wenn n_to_plot 1 ist
    if n_to_plot == 1:
        axes = axes[:, np.newaxis]

    for i in range(n_to_plot):
        perturbation = x_adv_filtered[i] - x_clean_filtered[i]

        axes[0, i].imshow(x_clean_filtered[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[0, i].set_title(class_names[y_true_filtered[i]], fontsize=9)
        axes[0, i].axis('off')

        # Speichere das Image-Objekt fuer die Colorbar
        im = axes[1, i].imshow(perturbation.squeeze(), cmap='seismic', vmin=-eps, vmax=eps)
        axes[1, i].axis('off')

        axes[2, i].imshow(x_adv_filtered[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        color = 'red' # Diese sind per Definition falsch klassifiziert
        axes[2, i].set_title(class_names[preds_adv_filtered[i]], fontsize=9, color=color)
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=10)
    axes[1, 0].set_ylabel('Stoerung', fontsize=10)
    axes[2, 0].set_ylabel('Adversarial', fontsize=10)

    # Colorbar fuer die Stoerung hinzufuegen
    # Die Colorbar wird der ersten Stoerung zugeordnet und horizontal ueber die Reihe platziert
    cbar = fig.colorbar(im, ax=axes[1, :].ravel().tolist(), orientation='horizontal', fraction=0.046, pad=0.04)
    cbar.set_label(f'Pixelwertänderung (Blau: Reduktion, Rot: Erhöhung; max={eps:.2f})', fontsize=9)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.5) # Add more vertical spacing
    plt.show()


plot_adversarial_examples(x_test_subset, x_adv_fgsm, y_test_subset, adv_preds, class_names, max_n_to_show=10)

## 5. Wie stark muss die Stoerung sein? (eps-Sweep)

Wir variieren `eps` und beobachten, wie die Accuracy einbricht. Das macht sichtbar, dass schon sehr kleine, kaum sichtbare Stoerungen ausreichen.

In [ ]:
def accuracy_under_fgsm(classifier, x, y, eps):
    """Berechnet die Modell-Accuracy unter einem FGSM-Angriff.

    Args:
        classifier: ART-KerasClassifier-Instanz des anzugreifenden
            Modells.
        x: Eingabebilder, Form (N, 28, 28, 1), Werte in [0, 1].
        y: Wahre Klassen-Indizes zu x.
        eps: Staerke der FGSM-Stoerung (L-unendlich-Norm).

    Returns:
        float: Anteil korrekt klassifizierter adversarialer Bilder.
    """
    attack = FastGradientMethod(estimator=classifier, eps=eps)
    x_adv = attack.generate(x=x)
    preds = np.argmax(classifier.predict(x_adv), axis=1)
    return np.mean(preds == y)


eps_values = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
accuracies = [accuracy_under_fgsm(classifier, x_test_subset, y_test_subset, e) for e in eps_values]

plt.figure(figsize=(8, 5))
plt.plot(eps_values, [a * 100 for a in accuracies], marker='o')
plt.xlabel('eps (Staerke der Stoerung)')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy des Baseline-Modells unter FGSM in Abhaengigkeit von eps')
plt.grid(True)
plt.show()

## 6. Staerkerer Angriff: Projected Gradient Descent (PGD)

FGSM macht nur *einen* Gradientenschritt. **PGD** wiederholt diesen Schritt mehrfach mit kleineren Schrittweiten und projiziert nach jedem Schritt zurueck in die zulaessige `eps`-Umgebung. PGD gilt als deutlich staerkerer Angriff als FGSM und wird haeufig als Referenz fuer *Worst-Case*-Robustheit verwendet.

In [ ]:
attack_pgd = ProjectedGradientDescent(
    estimator=classifier, eps=eps, eps_step=eps / 10, max_iter=40, num_random_init=1
)

x_adv_pgd = attack_pgd.generate(x=x_test_subset)
pgd_preds = np.argmax(classifier.predict(x_adv_pgd), axis=1)
pgd_acc = np.mean(pgd_preds == y_test_subset)

print(f"Accuracy unter FGSM (eps={eps}): {adv_acc * 100:.2f} %")
print(f"Accuracy unter PGD  (eps={eps}): {pgd_acc * 100:.2f} %")

## 7. Adversarial Training

Beim *Adversarial Training* trainieren wir ein Modell nicht nur auf sauberen Bildern, sondern (teilweise) auf live erzeugten adversarialen Bildern. Das Modell lernt so, robuster gegenueber genau der Art von Stoerung zu werden, mit der es trainiert wurde.

Wir starten dafuer bewusst mit einem **frisch initialisierten** Modell derselben Architektur (nicht dem bereits fertig trainierten Baseline-Modell), damit der Vergleich fair ist: Baseline-Training vs. Adversarial-Training, beide von Grund auf, gleiche Architektur, gleiche Anzahl Epochen.

In [ ]:
def build_model():
    """Baut ein frisches, unkompiliertes-dann-kompiliertes CNN fuer Fashion MNIST.

    Verwendet dieselbe Architektur wie das Baseline-Modell aus
    03_fashion_mnist_bias_variance.ipynb, damit der Vergleich zwischen
    Standard- und Adversarial-Training auf gleicher Grundlage steht.

    Returns:
        tf.keras.Model: Ein kompiliertes, aber noch untrainiertes CNN.
    """
    tf.random.set_seed(42)

    DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding='same',
                            activation='relu', kernel_initializer='he_normal')

    new_model = tf.keras.Sequential([
        DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPool2D(),
        DefaultConv2D(filters=128),
        DefaultConv2D(filters=128),
        tf.keras.layers.MaxPool2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation='relu',
                              kernel_initializer='he_normal'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation='softmax')
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)
    new_model.compile(loss='sparse_categorical_crossentropy',
                      optimizer=optimizer,
                      metrics=['accuracy'])
    return new_model

In [ ]:
robust_model = build_model()
robust_classifier = KerasClassifier(robust_model, clip_values=(0, 1), use_logits=False)

# Attacke, die WAEHREND des Trainings genutzt wird, um adversariale Batches zu erzeugen
attack_for_training = FastGradientMethod(estimator=robust_classifier, eps=eps)

# ratio=0.5: die Haelfte jeder Trainings-Batch besteht aus adversarialen Beispielen,
# die andere Haelfte aus sauberen Bildern
trainer = AdversarialTrainer(robust_classifier, attacks=attack_for_training, ratio=0.5)

trainer.fit(x_train, y_train, nb_epochs=10, batch_size=128)

## 8. Vergleich: robustes Modell vs. Baseline-Modell

Jetzt vergleichen wir beide Modelle -- auf sauberen Bildern und unter Angriff.

In [ ]:
def evaluate_clean_and_adversarial(classifier, x, y, eps):
    """Ermittelt die Accuracy eines Modells auf sauberen und adversarialen Bildern.

    Args:
        classifier: ART-KerasClassifier-Instanz des zu evaluierenden
            Modells.
        x: Eingabebilder, Form (N, 28, 28, 1), Werte in [0, 1].
        y: Wahre Klassen-Indizes zu x.
        eps: Staerke des FGSM-Angriffs zur Robustheitsmessung.

    Returns:
        tuple[float, float]: (Clean Accuracy, Accuracy unter FGSM-Angriff).
    """
    clean_p = np.argmax(classifier.predict(x), axis=1)
    clean_a = np.mean(clean_p == y)

    attack = FastGradientMethod(estimator=classifier, eps=eps)
    x_adv = attack.generate(x=x)
    adv_p = np.argmax(classifier.predict(x_adv), axis=1)
    adv_a = np.mean(adv_p == y)
    return clean_a, adv_a


baseline_clean, baseline_adv = evaluate_clean_and_adversarial(classifier, x_test_subset, y_test_subset, eps)
robust_clean, robust_adv = evaluate_clean_and_adversarial(robust_classifier, x_test_subset, y_test_subset, eps)

print(f"{'Modell':<20}{'Clean Acc.':>12}{'Acc. unter FGSM':>20}")
print(f"{'Baseline':<20}{baseline_clean*100:>11.2f}%{baseline_adv*100:>19.2f}%")
print(f"{'Adversarial-trained':<20}{robust_clean*100:>11.2f}%{robust_adv*100:>19.2f}%")

In [ ]:
labels_plot = ['Clean', 'Unter FGSM-Angriff']
baseline_vals = [baseline_clean * 100, baseline_adv * 100]
robust_vals = [robust_clean * 100, robust_adv * 100]

x_pos = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_pos - width/2, baseline_vals, width, label='Baseline-Modell', color='#ED7D31')
ax.bar(x_pos + width/2, robust_vals, width, label='Adversarial-trainiertes Modell', color='#00B050')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels_plot)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Baseline vs. Adversarial Training unter FGSM-Angriff')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Aufgabe fuer Studierende

1. Wiederholen Sie den `eps`-Sweep aus Abschnitt 5 fuer das adversarial-trainierte Modell. Bei welchem `eps` faengt die Robustheit an zusammenzubrechen -- im Vergleich zum Baseline-Modell?
2. Testen Sie das adversarial-trainierte Modell gegen PGD (nicht nur FGSM, mit dem es trainiert wurde). Ist es auch gegen diesen staerkeren Angriff robuster, oder nur gegen FGSM speziell?
3. Wie veraendert sich die Clean Accuracy durch das Adversarial Training? Gibt es einen Trade-off zwischen Robustheit und Genauigkeit auf sauberen Daten?
4. Experimentieren Sie mit dem `ratio`-Parameter des `AdversarialTrainer` (Anteil adversarialer Beispiele pro Batch). Wie wirkt sich das auf Robustheit und Clean Accuracy aus?